### Data Ingestion

In [2]:
### Document data structure
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader

C:\Users\Akhil\AppData\Local\Temp\ipykernel_30528\2762714175.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader


In [3]:
document = Document(
    page_content = "this is the main text content I am using to create RAG",
    metadata = {
        "source": "example.txt",
        "pages": 1,
        "author": "Akhil Abraham",\
        "date_created": "2026-06-28"
    }
)
document

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Akhil Abraham', 'date_created': '2026-06-28'}, page_content='this is the main text content I am using to create RAG')

In [4]:
# Create a simple text document
os.makedirs("../data/text_files", exist_ok = True)

simple_text = {
    "../data/text_files/python_introduction.txt": """Python Programming Introduction
    
    
    Python is a high-level, interpreted programming language known for its simplicity and readability.
    It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, 
    including procedural, object-oriented, and functional programming.
    
    Key Features:
    - Easy to learn and use
    - Extensive standard library
    - Cross-platform compatibility
    - Large community support
    
    Python is widely used in web development, data analysis, artificial intelligence, scientific computing, and more.""",
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

    Machine learning is a subset of artificial intelligence that focuses on building systems that can learn from data 
    and improve their performance over time without being explicitly programmed. It focuses on developing algorithms 
    that can identify patterns and make predictions based on data.

    Types of Machine Learning:
    1. Supervised Learning: The model is trained on labeled data, where the input-output pairs are provided.
    2. Unsupervised Learning: The model is trained on unlabeled data, and it tries to find hidden patterns or structures in the data.
    3. Reinforcement Learning: The model learns by interacting with an environment and receiving feedback in the form of rewards or penalties.

    Applications include image recognition, natural language processing, recommendation systems, and autonomous vehicles.""",
}

In [5]:
# Using Dictionary "simple_text" to create text files
for filepath, content in simple_text.items():
    with open(filepath, "w", encoding = "utf-8") as f:
        f.write(content)

print("Text files created successfully.")

Text files created successfully.


In [6]:
# Reading this text file using LangChain TextLoader
loader = TextLoader("../data/text_files/python_introduction.txt", encoding = "utf-8")
document = loader.load()

print(document)
print("Document loaded successfully.")

[Document(metadata={'source': '../data/text_files/python_introduction.txt'}, page_content='Python Programming Introduction\n\n\n    Python is a high-level, interpreted programming language known for its simplicity and readability.\n    It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, \n    including procedural, object-oriented, and functional programming.\n\n    Key Features:\n    - Easy to learn and use\n    - Extensive standard library\n    - Cross-platform compatibility\n    - Large community support\n\n    Python is widely used in web development, data analysis, artificial intelligence, scientific computing, and more.')]
Document loaded successfully.


In [7]:
# Reading text files using LangChain DirectoryLoader
dir_loader = DirectoryLoader("../data/text_files", glob = "*.txt", loader_cls = TextLoader, loader_kwargs = {"encoding": "utf-8"}, show_progress = True)

directory_documents = dir_loader.load()
print(directory_documents)
print("Documents loaded successfully from directory.")
print(f"Total documents loaded: {len(directory_documents)}")

100%|██████████| 2/2 [00:00<00:00, 1490.78it/s]

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\n    Machine learning is a subset of artificial intelligence that focuses on building systems that can learn from data \n    and improve their performance over time without being explicitly programmed. It focuses on developing algorithms \n    that can identify patterns and make predictions based on data.\n\n    Types of Machine Learning:\n    1. Supervised Learning: The model is trained on labeled data, where the input-output pairs are provided.\n    2. Unsupervised Learning: The model is trained on unlabeled data, and it tries to find hidden patterns or structures in the data.\n    3. Reinforcement Learning: The model learns by interacting with an environment and receiving feedback in the form of rewards or penalties.\n\n    Applications include image recognition, natural language processing, recommendation systems, and autonomous vehicles.'), Document(metadata={'sourc

In [8]:
# Reading pdf files using LangChain DirectoryLoader
dir_loader = DirectoryLoader("../data/pdf_files", glob = "*.pdf", loader_cls = PyMuPDFLoader, show_progress = True)

directory_pdf_documents = dir_loader.load()
print(directory_pdf_documents)
print("Documents loaded successfully from directory.")
print(f"Total documents loaded: {len(directory_pdf_documents)}")

100%|██████████| 2/2 [00:00<00:00,  3.94it/s]

[Document(metadata={'producer': 'xdvipdfmx (20250410)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-20T17:00:24+00:00', 'source': '..\\data\\pdf_files\\akhil_abraham_resume_2026.pdf', 'file_path': '..\\data\\pdf_files\\akhil_abraham_resume_2026.pdf', 'total_pages': 2, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260120170024Z', 'page': 0}, page_content='Akhil Abraham\nBelfast, Northern Ireland, UK\nĦ +44 7717112595\n|\nć akhilabrahamuk@gmail.com\n|\n] linkedin.com/in/akhil-abraham/\nPersonal Profile\nDataEngineerwithastrongtechnicalbackgroundandhands‑onexperiencedesigninganddeliveringscalable,efficientETLpipelines\nfor large‑scale data migration and transformation projects. Curious, proactive, and eager to learn, with a strong commitment\nto helping clients make better, insight‑driven decisions. Proficient in building and maintaining high‑quality data solutions using\nClov

In [9]:
type(directory_pdf_documents[0])

langchain_core.documents.base.Document

# Create Data Chunks

In [20]:
# import RecursiveCharacterTextSplitter for splitting documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )

    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [22]:
chunks = split_documents(directory_pdf_documents)

Split 11 documents into 6 chunks

Example chunk:
Content: Akhil Abraham
Belfast, Northern Ireland, UK
Ħ +44 7717112595
|
ć akhilabrahamuk@gmail.com
|
] linkedin.com/in/akhil-abraham/
Personal Profile
DataEngineerwithastrongtechnicalbackgroundandhands‑onexper...
Metadata: {'producer': 'xdvipdfmx (20250410)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-20T17:00:24+00:00', 'source': '..\\data\\pdf_files\\akhil_abraham_resume_2026.pdf', 'file_path': '..\\data\\pdf_files\\akhil_abraham_resume_2026.pdf', 'total_pages': 2, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260120170024Z', 'page': 0}


# Embedding and VectorStoreDB

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    """
    Handles embedding generation using Sentence Transformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the embedding manager
        
        Args:
            model_name : HuggingFace model name for embedding generation.
        """
        self.model_name = model_name
        self.model = None
        self.load_model() # it will load the model when the class is initialized

    def load_model(self):
        """
        Loads the embedding model from HuggingFace.
        """
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates embeddings for a list of texts.
        
        Args:
            texts : List of strings to generate embeddings for.
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dimension).
        """
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar =  True)
        print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
        return embeddings        

In [17]:
# Initialize the EmbeddingManager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2763.63it/s]


Embedding model loaded successfully. Embedding dimension: 384


In [18]:
# VectorStoreManager class to handle vector store operations
class VectorStoreManager:
    """Manages vector store operations using ChromaDB."""

    def __init__(self, collection_name: str = "pdf_embeddings", persist_directory: str = "../data/vector_store"):
        """
        Initializes the vector store manager.
        
        Args:
            collection_name : Name of the ChromaDB collection.
            persist_directory : Directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            # Create the persist directory
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(name = self.collection_name, metadata = {"description": "PDF Embeddings Collection for RAG Project"})
            print(f"Vector store initialized successfully. Collection name: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adds documents and their embeddings to the vector store.
        
        Args:
            documents : List of LangChain documents.
            embeddings : Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store.")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        document_texts = []
        embedding_list = []

        for i, (document, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(document.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(document.page_content)
            metadatas.append(metadata)

            # Document context
            document_texts.append(document.page_content)

            # Prepare embedding
            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                documents = document_texts,
                embeddings = embedding_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [19]:
# Initialize the VectorStoreManager
vector_store_manager = VectorStoreManager()
vector_store_manager

Vector store initialized successfully. Collection name: pdf_embeddings
Existing documents in the collection: 0


In [23]:
chunks

[Document(metadata={'producer': 'xdvipdfmx (20250410)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-20T17:00:24+00:00', 'source': '..\\data\\pdf_files\\akhil_abraham_resume_2026.pdf', 'file_path': '..\\data\\pdf_files\\akhil_abraham_resume_2026.pdf', 'total_pages': 2, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260120170024Z', 'page': 0}, page_content='Akhil Abraham\nBelfast, Northern Ireland, UK\nĦ +44 7717112595\n|\nć akhilabrahamuk@gmail.com\n|\n] linkedin.com/in/akhil-abraham/\nPersonal Profile\nDataEngineerwithastrongtechnicalbackgroundandhands‑onexperiencedesigninganddeliveringscalable,efficientETLpipelines\nfor large‑scale data migration and transformation projects. Curious, proactive, and eager to learn, with a strong commitment\nto helping clients make better, insight‑driven decisions. Proficient in building and maintaining high‑quality data solutions using\nClov